In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

import os
import time
import numpy as np
from itertools import chain
import pickle
from glob import glob
from pathlib import Path

from multiprocessing import Pool

from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from utils.write_output import write_array2txt

from cylinder_extraction.utils import extract_cylinders

# Read point cloud

In [2]:
data_dir = Path('D:/REGISTRATION DATA/External/JB-550 90ft EXT')
filename_template = '{}.e57'
ptc_col_names =  ["x","y","z"]


In [7]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
stations = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    fn = Path(fp).stem
    print(f'STATION {fn}')
    # station id
    sn = fn[-11:-8]
    station_names.append(sn)
    ptc_fn = filename_template.format(sn)
    print(ptc_fn)
    station = Station(
        name=sn, file_path=fp, read_e57_raw=False
    )
    stations.append(station)

with Pool(12) as p:
    stations = p.map(
        extract_cylinders,
        stations
    )

STATION 3701_Station 001_Scan 01
001.e57
D:\REGISTRATION DATA\External\JB-550 90ft EXT 3701_Station 001_Scan 01.e57
0
['sphericalRange', 'sphericalAzimuth', 'sphericalElevation', 'intensity', 'colorRed', 'colorGreen', 'colorBlue', 'rowIndex', 'columnIndex']
[]
555560076
{}


E57Exception: bad API function argument provided by user (E57_ERROR_BAD_API_ARGUMENT)

In [10]:
for station in stations:
    # write clusters
    for cltr in station.clusters:
        write_array2txt(
            cltr['filepath'],
            cltr['points']
        )
    station.clusters = None
    # write station metadata
    out_dir = os.path.join(data_dir,station.name + '-output')
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    # save station without points
    station.ptc = None
    out_fp = os.path.join(out_dir, f'{station.name}.pkl')
    with open(out_fp, 'wb') as f:
        pickle.dump(station, f)

In [ ]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    fn = Path(fp).stem
    print(f'STATION {fn}')
    # station id
    sn = fn.split('_')[-1]
    station_names.append(sn)
    ptc_fn = filename_template.format(sn)
    print(ptc_fn)
    station = Station(
        name=sn, file_path=fp
    )
    # station.write_points()
    station.extract_cylinder(refine=True)
    out_dir = os.path.join(data_dir,station.name + '-output')
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    # save station without points
    station.ptc = None
    out_fp = os.path.join(out_dir, f'{station.name}.pkl')
    with open(out_fp, 'wb') as f:
        pickle.dump(station, f)
# # data/JB-483/Jb-483 Scan_SuncorTk20D71Scan_055.e57
# for sn in station_names:
#     ptc_fn = filename_template.format(sn)
#     print(ptc_fn)
#     station = Station(
#         name=sn, file_path=os.path.join(data_dir, ptc_fn)
#     )
#     # station.write_points()
#     station.extract_cylinder()
#     out_dir = os.path.join(data_dir,station.name + '-output')
#     if not os.path.exists(out_dir):
#         os.mkdir(out_dir)
#     out_fp = os.path.join(out_dir, f'{station.name}.pkl')
#     with open(out_fp, 'wb') as f:
#         pickle.dump(station, f)